# Querying the Level 1 Iceberg catalog

This notebook reads the GNSS-A Level 1 tables written by the Go ingest in `src/go`:

```
sfg level1 ingest --data <.../network/station/campaign/raw> --warehouse <WAREHOUSE>
```

The catalog is a SQLite file (`catalog.db`) inside the warehouse folder, and the table data is stored beside it. Level 1 tables live in the `gnssa_l1` namespace and are partitioned by `Network` / `Station` / `Campaign`.

`gnss_observations` is the exception: it is the gnsstools observations table, created from gnsstools' own definition. It has no scope or `Log_Source` columns. Each row's `stream_edid` (`network:station:campaign:filetype:logsource`) carries both, and the table is partitioned by GPS day and a hash bucket of `stream_edid`.

| Table | From | Time column |
|---|---|---|
| `gnss_observations` | NOV000 (`RANGE`), NOV770 (`RANGECMP5B`, `RANGECMP5B_1`); gnsstools schema | `time_gpst` (GPS wall clock), `time_utc` |
| `inspva`, `insstdev` | NOV000 | `time` (UTC) |
| `attitude_ahrs` | AHRS00, DFOP00 | `time` (UTC) |
| `travel_times` | DFOP00 | `T_transmit`, `T_receive` (UTC) |
| `sound_speed` | CTD / Sea-Bird (no parser yet) | `time` (UTC) |

In [ ]:
import os
from datetime import datetime, timedelta, timezone

import matplotlib.pyplot as plt
import pandas as pd
import pyarrow.compute as pc
from pyiceberg.catalog import load_catalog
from pyiceberg.expressions import AlwaysTrue, And, EqualTo, GreaterThanOrEqual, LessThan, StartsWith

# Warehouse written by `go run .` in dev/level1; override with SFG_ICEBERG_WAREHOUSE.
WAREHOUSE = os.environ.get(
    "SFG_ICEBERG_WAREHOUSE", "/Users/franklyndunbar/Project/SeaFloorGeodesy/Data/ICEBERG"
)
NAMESPACE = "gnssa_l1"

catalog = load_catalog(
    "default",
    type="sql",
    uri=f"sqlite:///{WAREHOUSE}/catalog.db",
    warehouse=f"file://{WAREHOUSE}",
)
catalog.list_tables(NAMESPACE)

## What's in the catalog

Row counts come from table metadata (snapshot summaries and partition stats), so these cells don't read any data files.

In [ ]:
def table(name):
    return catalog.load_table(f"{NAMESPACE}.{name}")


def overview():
    rows = []
    for _, name in catalog.list_tables(NAMESPACE):
        t = table(name)
        snap = t.current_snapshot()
        rows.append({
            "table": name,
            "rows": int(snap.summary.additional_properties.get("total-records", 0)) if snap else 0,
            "snapshots": len(t.snapshots()),
            "partitioned_by": ", ".join(f.name for f in t.spec().fields),
        })
    return pd.DataFrame(rows).set_index("table")


overview()

In [ ]:
def inventory(name):
    """Rows and files per partition (scope; GPS day and stream_edid bucket for gnss_observations)."""
    t = table(name)
    if t.current_snapshot() is None:
        return pd.DataFrame()
    parts = t.inspect.partitions().to_pylist()
    df = pd.DataFrame([{**p["partition"], "rows": p["record_count"], "files": p["file_count"]} for p in parts])
    return df.sort_values(list(df.columns[:-2])).reset_index(drop=True)


inventory("attitude_ahrs")

In [ ]:
def describe(name):
    """Columns with type, requiredness and doc."""
    t = table(name)
    print(f"{NAMESPACE}.{name}")
    print("  partition spec:", t.spec())
    print("  sort order:   ", t.sort_order())
    return pd.DataFrame(
        [{"column": f.name, "type": str(f.field_type), "required": f.required, "doc": f.doc}
         for f in t.schema().fields]
    )


describe("travel_times")

## Reading rows

`read()` builds an Iceberg row filter from the scope and an optional time window, so only matching partitions and files get opened. Pass `columns` to read only what you need. `gnss_observations` is large: tens of millions of rows per campaign.

Times are given in UTC. For `gnss_observations` the window is applied to `time_gpst`, the partition column, which holds the GPS wall clock, so `read()` shifts the window by the GPS−UTC leap offset (18 s since 2017).

For `gnss_observations` the scope becomes a `stream_edid` prefix (`network:station:campaign:`), so it can't skip a leading part: give `network` to filter by `station`, and so on. Pass `filetype`/`logsource` to pick a single stream. Only the GPS-day partition prunes files there, so keep time windows tight.

In [ ]:
GPS_UTC_LEAP = timedelta(seconds=18)  # valid for epochs since 2017-01-01

# Column each table's time window filters on.
TIME_COLUMN = {"gnss_observations": "time_gpst", "travel_times": "T_transmit"}


def _ts(t):
    if isinstance(t, str):
        t = datetime.fromisoformat(t)
    if t.tzinfo is None:
        t = t.replace(tzinfo=timezone.utc)
    return t


def _stream_prefix(*parts):
    """stream_edid prefix from leading parts (network, station, campaign, filetype, logsource)."""
    given = [p for p in parts if p is not None]
    if parts[: len(given)] != tuple(given):
        raise ValueError("gnss_observations scope must be given from network down, without gaps")
    if not given:
        return None
    full = len(given) == len(parts)
    return ":".join(given) + ("" if full else ":"), full


def read(name, network=None, station=None, campaign=None, start=None, end=None,
         columns=None, where=None, limit=None, filetype=None, logsource=None):
    """Read rows of a Level 1 table as a DataFrame.

    start/end are UTC (datetime or ISO string), start inclusive, end exclusive.
    where is an extra pyiceberg expression, e.g. EqualTo("Log_Source", "AHRS00").
    filetype/logsource only apply to gnss_observations (stream_edid parts).
    """
    expr = AlwaysTrue()
    if name == "gnss_observations":
        prefix = _stream_prefix(network, station, campaign, filetype, logsource)
        if prefix:
            value, full = prefix
            expr = EqualTo("stream_edid", value) if full else StartsWith("stream_edid", value)
    else:
        for col, val in (("Network", network), ("Station", station), ("Campaign", campaign)):
            if val is not None:
                expr = And(expr, EqualTo(col, val))
    tcol = TIME_COLUMN.get(name, "time")
    shift = GPS_UTC_LEAP if tcol == "time_gpst" else timedelta(0)
    if start is not None:
        expr = And(expr, GreaterThanOrEqual(tcol, (_ts(start) + shift).isoformat()))
    if end is not None:
        expr = And(expr, LessThan(tcol, (_ts(end) + shift).isoformat()))
    if where is not None:
        expr = And(expr, where)
    scan = table(name).scan(
        row_filter=expr,
        selected_fields=tuple(columns) if columns else ("*",),
        limit=limit,
    )
    return scan.to_pandas()

In [ ]:
def streams(start=None, end=None):
    """Rows per stream_edid in gnss_observations, split into its parts.

    File metadata truncates stream_edid, so this reads the stream_edid column;
    pass a UTC start/end to limit it to some GPS days.
    """
    df = read("gnss_observations", start=start, end=end, columns=["stream_edid"])
    counts = df["stream_edid"].value_counts().rename("rows").reset_index()
    parts = counts["stream_edid"].str.split(":", expand=True)
    parts.columns = ["network", "station", "campaign", "filetype", "logsource"]
    return pd.concat([parts, counts["rows"]], axis=1).sort_values(list(parts.columns)).reset_index(drop=True)


streams()

Pick a scope from the inventory. By default this is the first `inspva` partition. The example window starts at its first INS position. Before the INS has a position, INSPVA logs all-zero rows (lat = lon = 0), and those rows are skipped here.

In [ ]:
inv = inventory("inspva")
first_part = inv.iloc[0]
scope = dict(network=first_part["Network"], station=first_part["Station"], campaign=first_part["Campaign"])
print(scope)

span = read("inspva", **scope, columns=["time", "lat", "lon"])
valid = span[(span["lat"] != 0) | (span["lon"] != 0)]["time"]
t0, t1 = valid.min(), valid.max()
print(f"INS positions {t0} .. {t1} ({len(valid):,} of {len(span):,} epochs non-zero)")
window = dict(start=t0, end=t0 + timedelta(minutes=10))

## INS position and attitude (`inspva`, `insstdev`)

In [ ]:
pva = read("inspva", **scope, **window)
std = read("insstdev", **scope, **window)

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(pva["lon"], pva["lat"], lw=0.8)
ax[0].set(xlabel="lon [deg]", ylabel="lat [deg]", title="track")
for c in ["roll", "pitch"]:
    ax[1].plot(pva["time"], pva[c], label=c, lw=0.8)
ax[1].set(title="attitude [deg]"); ax[1].legend()
for c in ["lat_sig", "lon_sig", "alt_sig"]:
    ax[2].plot(std["time"], std[c], label=c, lw=0.8)
ax[2].set(title="position std dev [m]"); ax[2].legend()
fig.autofmt_xdate(); plt.tight_layout()
pva.head()

## Attitude (`attitude_ahrs`)

AHRS00 is the full-rate XSENS log. DFOP00 rows come from the AHRS block of SV3 events, are much sparser, and carry `time_start_count`/`time_status`. Some rows report only magnetic heading (`h_mag`), so the other measurement columns are null on those rows.

In [ ]:
att = read("attitude_ahrs", **scope, columns=["time", "Log_Source", "h", "h_mag", "p", "r"])
print(att.groupby("Log_Source")["time"].agg(["count", "min", "max"]))

first = att[att["Log_Source"] == "AHRS00"].sort_values("time").head(20_000)
fig, ax = plt.subplots(figsize=(12, 3.5))
for c in ["h", "h_mag", "p", "r"]:
    ax.plot(first["time"], first[c], label=c, lw=0.6)
ax.set(ylabel="deg", title="AHRS00 attitude (first 20k samples)"); ax.legend()
fig.autofmt_xdate()

## GNSS observations (`gnss_observations`)

One row per (epoch, satellite, signal), in the gnsstools observations schema. `stream_edid` is `network:station:campaign:filetype:logsource`. Its last part is the NovAtel log, which keeps the NOV770 antennas apart (`RANGECMP5B` is the primary antenna, `RANGECMP5B_1` the secondary). The cells below split it out as a `logsource` column.

In [ ]:
def with_logsource(df):
    return df.assign(logsource=df["stream_edid"].str.rsplit(":", n=1).str[-1])


obs = with_logsource(read(
    "gnss_observations", **scope, start=t0, end=t0 + timedelta(minutes=1),
    columns=["stream_edid", "time_utc", "constellation", "satellite_id", "signal", "code", "phase", "snr"],
))
print(f"{len(obs):,} observations in 1 minute")
obs.groupby(["logsource", "constellation"]).agg(
    satellites=("satellite_id", "nunique"),
    signals=("signal", lambda s: ", ".join(sorted(s.unique()))),
    mean_snr=("snr", "mean"),
)

In [ ]:
# Satellites tracked per epoch, per stream.
per_epoch = (obs.groupby(["logsource", "time_utc"])["satellite_id"]
                .nunique().rename("satellites").reset_index())
fig, ax = plt.subplots(figsize=(12, 3.5))
for src, g in per_epoch.groupby("logsource"):
    ax.plot(g["time_utc"], g["satellites"], label=src, lw=0.8)
ax.set(ylabel="satellites", title="satellites per epoch"); ax.legend()
fig.autofmt_xdate()

In [ ]:
# SNR of one GPS satellite over a longer window, read with a pushed-down filter.
sv = obs.query("constellation == 'G'")["satellite_id"].mode().iloc[0]
one = with_logsource(read(
    "gnss_observations", **scope, start=t0, end=t0 + timedelta(minutes=30),
    columns=["stream_edid", "time_utc", "signal", "snr"],
    where=And(EqualTo("constellation", "G"), EqualTo("satellite_id", int(sv))),
))
fig, ax = plt.subplots(figsize=(12, 3.5))
for (src, sig), g in one.groupby(["logsource", "signal"]):
    ax.plot(g["time_utc"], g["snr"], label=f"{src} {sig}", lw=0.7)
ax.set(ylabel="dB-Hz", title=f"G{sv:02d} SNR"); ax.legend(fontsize=8)
fig.autofmt_xdate()

## Two-way travel times (`travel_times`)

`TravelTime` and `tat` are stored raw, as logged. The trigger-delay correction is applied when Level 2 is built. A DFOP00 file with no transponder replies (all ranges 0) gives no rows.

In [ ]:
tt = read("travel_times", **scope)
print(f"{len(tt):,} travel times")
tt.groupby("MT_ID")["TravelTime"].describe() if len(tt) else tt

## Table history

Each append is an Iceberg snapshot, so you can see what was loaded when, or read the table as of an earlier snapshot.

In [ ]:
t = table("attitude_ahrs")
pd.DataFrame(
    [{"snapshot_id": s.snapshot_id,
      "committed": datetime.fromtimestamp(s.timestamp_ms / 1000, tz=timezone.utc),
      "added_rows": int(s.summary.additional_properties.get("added-records", 0)),
      "total_rows": int(s.summary.additional_properties.get("total-records", 0))}
     for s in t.snapshots()]
)